In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx

import sempipes
from sempipes.optimisers.trajectory import load_trajectory_from_json

# optimise_colopro saves a trajectory to `.sempipes_trajectories/` in the working directory it was run from.
# We load the most recent one, looking in the current folder and its parents (e.g. the repository root).
search_dirs = [Path.cwd(), *list(Path.cwd().parents)[:3]]
trajectory_files = sorted(
    (path for folder in search_dirs for path in (folder / ".sempipes_trajectories").glob("*.json")),
    key=lambda path: path.stat().st_mtime,
)
assert trajectory_files, "No trajectory found - run examples/12_optimise_colopro.py or an optimisation notebook first."
trajectory_path = trajectory_files[-1]
print(f"Loading {trajectory_path}")
trajectory = load_trajectory_from_json(str(trajectory_path))

scores = {}
edges = []

for outcome in trajectory.outcomes:
    node_id = outcome.search_node.trial
    if outcome.search_node.parent_trial is not None:
        edges.append((outcome.search_node.parent_trial, node_id))
    scores[node_id] = outcome.score

G = nx.DiGraph()
G.add_nodes_from(scores)
G.add_edges_from(edges)

# Identify the node with the highest score (higher is better in sempipes)
best_node = max(scores, key=lambda node: scores[node])
print(f"Best trial: {best_node}, score: {scores[best_node]:.3f}")


def tree_layout(graph):
    """Simple top-down layout: one row per depth in the search tree (no Graphviz needed)."""
    roots = [node for node in graph.nodes if graph.in_degree(node) == 0]
    depth = {}
    for root in roots:
        for node, distance in nx.single_source_shortest_path_length(graph, root).items():
            depth[node] = min(distance, depth.get(node, distance))
    positions = {}
    for level in sorted(set(depth.values())):
        nodes_at_level = sorted(node for node in depth if depth[node] == level)
        for i, node in enumerate(nodes_at_level):
            positions[node] = (i - (len(nodes_at_level) - 1) / 2, -level)
    return positions


node_colors = ["red" if node == best_node else "white" for node in G.nodes()]
labels = {node: f"{node}\n{scores[node]:.3f}" for node in G.nodes()}

plt.figure(figsize=(12, 6))
nx.draw(
    G,
    tree_layout(G),
    with_labels=True,
    labels=labels,
    arrows=True,
    node_size=900,
    node_color=node_colors,
    edgecolors="black",
    font_size=7,
)
plt.title("Search tree of the optimisation (label: trial / score, red: best)")
plt.show()

In [2]:
best_outcome = max(trajectory.outcomes, key=lambda x: (x.score, -x.search_node.trial))

In [ ]:
# The best outcome stores the state (e.g. the generated code) for every optimised operator
for operator_name in best_outcome.states.operators():
    print(f"#### Operator: {operator_name}")
    print(best_outcome.states.get(operator_name).get("generated_code"))

In [7]:
trajectory.sempipes_config

Config(llm_for_code_generation=LLM(name='gemini/gemini-2.5-flash', parameters={'temperature': 2.0}), llm_for_batch_processing=LLM(name='ollama/gpt-oss:120b', parameters={'api_base': 'http://localhost:11434', 'temperature': 0.0}), batch_size_for_batch_processing=20, prefer_empty_state_in_preview=True)